In [1]:
import pandas as pd
import sqlite3

conn=sqlite3.connect('bi_warehouse.db')

## query 1: Total Sales per category per year
q1 = """
SELECT p.category, d.year, 
       ROUND(SUM(f.sales_amount), 2) AS total_sales,
       ROUND(SUM(f.profit), 2) AS total_profit
FROM fact_sales f
JOIN dim_product p ON f.product_key = p.product_key
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY p.category, d.year
ORDER BY d.year, total_sales DESC
"""


q2=""" 
SELECT c.segemnt, c.customer_name,
Round(SUM(f.sales_amount),2) as Revenue
FROM fact_sales f
JOIN dim_customer c on c.customer_key = f.customer_key
GROUP BY c.customer_name, c.segemnt
ORDER BY Revenue desc LIMIT 5"""

q3= """
SELECT r.region, d.year, d.quarter,
    ROUND(SUM(f.sales_amount),2) as Total_Sales
FROM fact_sales f
JOIN dim_region r on r.region_key=f.region_key
JOIN dim_date d on d.date_key=f.date_key
GROUP BY r.region, d.year, d.quarter
ORDER BY d.year, d.quarter
"""

q4="""
SELECT p.sub_category,
    ROUND(SUM(f.sales_amount),2) as Total_sales,
    ROUND(SUM(profit),2) as profit,
    ROUND(SUM(f.profit)*100/SUM(f.sales_amount),1) AS Profit_Margin
FROM fact_sales f
JOIN dim_product p on p.product_key=f.product_key
GROUP BY p.sub_category
ORDER BY Profit_Margin DESC """



for name, q in [("Sales by Category/Year", q1), ("Top Customer",q2),
                ("Sales by region and quarter",q3), ("Profit Margin",q4)]:
    print(f"\n=== {name} ===")
    print(pd.read_sql(q, conn).to_string(index=False))
print("Queries Executed Successfully.")

conn.close()


=== Sales by Category/Year ===
       category  year  total_sales  total_profit
     Technology  2014    249777.43      30730.45
      Furniture  2014    205737.52       6218.67
Office Supplies  2014    194331.16      32632.51
      Furniture  2015    237463.17       5686.15
     Technology  2015    237145.77      50109.40
Office Supplies  2015    195506.73      39475.93
     Technology  2016    337096.47      52763.37
      Furniture  2016    287349.04      14084.56
Office Supplies  2016    254865.36      53808.55
     Technology  2017    403648.73      60848.21
Office Supplies  2017    360494.11      61989.55
      Furniture  2017    300203.86       3223.78

=== Top Customer ===
    segemnt  customer_name  Revenue
Home Office    Sean Miller 49300.23
  Corporate   Tamara Chand 38119.53
  Corporate   Bill Shonely 29519.54
  Corporate Grant Thornton 26844.50
  Corporate     Brian Moss 26197.33

=== Sales by region and quarter ===
 region  year  quarter  Total_Sales
Central  2014       

In [2]:
conn=sqlite3.connect("bi_warehouse.db")
for table in ['fact_sales','dim_date','dim_customer','dim_product','dim_region']:
    pd.read_sql(f"SELECT * FROM {table}",conn).to_csv(f"{table}.csv",index=False)
conn.close()